In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, confusion_matrix
import torch

In [3]:
# Data preparation for all models

columns = ["duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes", "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in", "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations", "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login", "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count", "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate", "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty"]
df = pd.read_csv("nsl-kdd/KDDTrain+_20Percent.txt", header=None, names = columns)

In [ ]:
y = (df["label"] != "normal").astype(int).values
# y
x = df.drop(columns=["label", "difficulty"])
x = pd.get_dummies(x, columns=["protocol_type", "service", "flag"])
# x

,duration,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,...,flag_REJ,flag_RSTO,flag_RSTOS0,flag_RSTR,flag_S0,flag_S1,flag_S2,flag_S3,flag_SF,flag_SH
0,0,491,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False
1,0,146,0,0,0,0,0,0,0,0,...,False,False,False,False,False,False,False,False,True,False
2,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False
3,0,232,8153,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
4,0,199,420,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25187,0,0,0,0,0,0,0,0,0,0,...,False,True,False,False,False,False,False,False,False,False
25188,0,334,0,0,0,0,0,0,1,0,...,False,False,False,False,False,False,False,False,True,False
25189,0,0,0,0,0,0,0,0,0,0,...,True,False,False,False,False,False,False,False,False,False
25190,0,0,0,0,0,0,0,0,0,0,...,False,False,False,False,True,False,False,False,False,False


In [ ]:
scalar = StandardScaler()
X_scaled = scalar.fit_transform(x)
# X_scaled

array([[-0.11355066, -0.00988885, -0.03930979, ..., -0.02440864,
         0.82613265, -0.04134984],
       [-0.11355066, -0.01003196, -0.03930979, ..., -0.02440864,
         0.82613265, -0.04134984],
       [-0.11355066, -0.01009252, -0.03930979, ..., -0.02440864,
        -1.21045936, -0.04134984],
       ...,
       [-0.11355066, -0.01009252, -0.03930979, ..., -0.02440864,
        -1.21045936, -0.04134984],
       [-0.11355066, -0.01009252, -0.03930979, ..., -0.02440864,
        -1.21045936, -0.04134984],
       [-0.11355066, -0.01009252, -0.03930979, ..., -0.02440864,
        -1.21045936, -0.04134984]])

In [8]:
pca = PCA(n_components=0.95, random_state=42)
X_pca = pca.fit_transform(X_scaled)
print(f"Final shape {X_pca.shape}")

Final shape (25192, 84)


In [9]:
num_nodes = len(X_pca)
indices = np.random.permutation(num_nodes)

In [10]:
train_size = int(0.7 * num_nodes)
val_size = int(0.15 * num_nodes)

In [11]:
train_mask = np.zeros(num_nodes, dtype=bool)
test_mask = np.zeros(num_nodes, dtype=bool)
val_mask = np.zeros(num_nodes, dtype=bool)

In [12]:
train_mask[indices[:train_size]] = True
test_mask[indices[train_size: train_size + val_size]] = True
val_mask[indices[train_size+val_size:]] = True

In [13]:
X_train, y_train = X_pca[train_mask], y[train_mask]
X_test, y_test = X_pca[test_mask], y[test_mask]
X_val, y_val = X_pca[val_mask], y[val_mask]

In [14]:
def evaluate_and_plot(y_true, y_pred, model, configs):
    print(f"{'-'*50}")
    print(f"Model: {model}")
    print(f"Hyper Parameters: {configs}" )
    print(f"{'-'*50}")

    print(classification_report(y_true, y_pred, target_names=["Normal", "Attack"]))

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Normal', 'Attack'], 
                yticklabels=['Normal', 'Attack'])
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.title(f"{model}")

    plt.show()
    plt.close()

In [ ]:
from sklearn.svm import SVC

svm_config = {
    'kernel': 'rbf',
    'C': 15, #Regularization param
    'class_weight': 'balanced' # adjusts weights inversely proportional to class frequencies
}

print("SVM Model")
svm_model = SVC(
    kernel=svm_config['kernel'],
    C=svm_config['C'],
    class_weight=svm_config['class_weight'],
    random_state=42
)

svm_model.fit(X_train, y_train)
svm_preds = svm_model.predict(X_test)
evaluate_and_plot(y_test, svm_preds, "SVM", svm_config)